# Healthcare Outcome Prediction — Disease Progression

End-to-end regression workflow using the scikit-learn diabetes dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
RANDOM_STATE = 42

## 1. Load data

In [ ]:
df = pd.read_csv("../data/diabetes_progression.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
display(df.describe().T)
display(df.isna().sum().to_frame("missing_values"))

## 2. Exploratory data analysis

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df.corr(numeric_only=True), cmap="vlag", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["target"], kde=True, ax=axes[0])
axes[0].set_title("Target Distribution")
sns.scatterplot(data=df, x="bmi", y="target", alpha=0.6, ax=axes[1])
axes[1].set_title("BMI vs Disease Progression")
plt.tight_layout()
plt.show()

## 3. Train/test split and leakage-safe preprocessing

In [ ]:
X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

## 4. Train and compare models

In [ ]:
models = {
    "ElasticNet": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=10000, random_state=RANDOM_STATE))
    ]),
    "GradientBoosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(n_estimators=250, learning_rate=0.03, max_depth=2, random_state=RANDOM_STATE))
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBRegressor(n_estimators=300, learning_rate=0.03, max_depth=3, subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=2))
    ])
}

rows = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred)
    })
metrics = pd.DataFrame(rows).sort_values("RMSE")
metrics

## 5. Cross-validation

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    scores = np.sqrt(-cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_mean_squared_error"))
    cv_rows.append({"Model": name, "CV_RMSE_Mean": scores.mean(), "CV_RMSE_SD": scores.std()})
pd.DataFrame(cv_rows).sort_values("CV_RMSE_Mean")

## 6. SHAP interpretability

In [ ]:
import shap
xgb_pipe = models["XGBoost"]
xgb_model = xgb_pipe.named_steps["model"]
X_test_imp = xgb_pipe.named_steps["imputer"].transform(X_test)
shap_explainer = shap.TreeExplainer(xgb_model)
shap_values = shap_explainer.shap_values(X_test_imp)
shap.summary_plot(shap_values, X_test_imp, feature_names=X.columns, max_display=10)

## 7. LIME local explanation

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
X_train_imp = xgb_pipe.named_steps["imputer"].transform(X_train)
lime_explainer = LimeTabularExplainer(
    X_train_imp, feature_names=list(X.columns), mode="regression", random_state=RANDOM_STATE
)
exp = lime_explainer.explain_instance(X_test_imp[0], xgb_model.predict, num_features=10)
exp.show_in_notebook(show_table=True)

## 8. Subgroup performance

This is an educational benchmark analysis. The groups below are derived from standardized age and sex variables and should not be interpreted as clinical fairness certification.

In [ ]:
best_model_name = metrics.iloc[0]["Model"]
best_pred = predictions[best_model_name]
test = X_test.copy()
test["actual"] = y_test.values
test["prediction"] = best_pred
test["sex_group"] = np.where(test["sex"] >= 0, "Sex >= 0", "Sex < 0")
test["age_group"] = np.where(test["age"] >= 0, "Age >= 0", "Age < 0")

for col in ["sex_group", "age_group"]:
    print("\n", col)
    display(test.groupby(col).apply(lambda g: pd.Series({
        "N": len(g),
        "MAE": mean_absolute_error(g["actual"], g["prediction"]),
        "RMSE": np.sqrt(mean_squared_error(g["actual"], g["prediction"])),
        "R2": r2_score(g["actual"], g["prediction"]) if len(g) > 1 else np.nan
    }), include_groups=False))

## 9. Conclusion

Select the model using both hold-out and cross-validation performance. Use SHAP for global feature importance and LIME for individual predictions. For real healthcare deployment, external validation, calibration, privacy review, clinical oversight, and formal fairness assessment would be required.